In [1]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 1. RNN-Based Text Generation

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import re
import requests
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Dataset:
url = "https://www.gutenberg.org/files/11/11-0.txt"
raw_text = requests.get(url).text
raw_text = re.sub(r'\s+', ' ', raw_text)  # Replace multiple whitespace with a single space
raw_text = raw_text.strip()  # Remove leading and trailing whitespace
raw_text = raw_text[:10000]  # Use only the first 10,000 characters for this example
print(len(raw_text), raw_text[:500])

specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
end_punctuations =  [".", "!", "?"] # Common sentence-ending punctuation marks

# Tokenization (with sentence boundaries):
def tokenize_with_sentence_boundaries(text):
    base_tokens = re.findall(r"\w+|[^\w\s*]", text)
    tokens = ["<sos>"]
    for token in base_tokens:
        tokens.append(token)
        if token in end_punctuations:
            tokens.append("<eos>")
            tokens.append("<sos>")
    if tokens and tokens[-1] == "<sos>":
            tokens.pop()
    if tokens and tokens[-1] != "<eos>":
            tokens.append("<eos>")
    return tokens


processed_text = tokenize_with_sentence_boundaries(raw_text) 
print(f"Number of tokens: {len(processed_text)}, First 500 characters: {processed_text[:500]}")

def encoder(tokens):
    vocab = sorted(set(specials + tokens))
    vocab = {token: idx for idx , token in enumerate(vocab)}
    encoded = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    return encoded, vocab


encoded_text, vocab = encoder(processed_text)
print(f"Vocabulary size: {len(vocab)}, \nFirst 100 tokens in vocabulary: {list(vocab.keys())[:100]} , \nFirst 100 encoded tokens: {encoded_text[:100]}")

# Dataset and DataLoader:
class TextDataset(Dataset):
    def __init__(self, encoded_text, seq_length):
        self.encoded_text = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded_text) - self.seq_length

    def __getitem__(self, idx):
        input_seq = self.encoded_text[idx:idx + self.seq_length]
        target_seq = self.encoded_text[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)
    
dataset = TextDataset(encoded_text, seq_length=50)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Hyperparameters:
embedding_dim = 128
hidden_dim = 256
vocab_size = len(vocab)
model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# Embedding Layer:
embedding_layer = nn.Embedding(vocab_size, embedding_dim)



Device: cpu
10000 *** START OF THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 Contents CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster
Number of tokens: 2509, First 500 characters: ['<sos>', 'START', 'OF', 'THE', 'PROJECT', 'GUTENBERG', 'EBOOK', '11', '[', 'Illustration', ']', 'Alice', '’', 's', 'Adventures', 'in', 'Wonderland', 'by', 'Lewis', 'Carroll', 'THE', 'MILLENNIUM', 'FULCRUM', 'EDITION', '3', '.', '<eos>', '<sos>', '0', 'Contents', 'CHAPTER', 'I', '.', '<eos>', '<sos>', 'Down', 'the', 'Rabbit', '-', 'Hole', 'CHAPTER', 'II', '.', '<eos>', '<sos>', 'The', 'Pool', 'of', 'Tears', 'CHAPTER', 'III', '.', '

NameError: name 'RNNLanguageModel' is not defined

### 1.a. Basic RNN Model:

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_layer):
        super().__init__()
        self.embedding = embedding_layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out)

model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


# Training Loop:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


Epoch 1/5, Loss: 3.9767
Epoch 2/5, Loss: 1.2686
Epoch 3/5, Loss: 0.3712
Epoch 4/5, Loss: 0.1829
Epoch 5/5, Loss: 0.1242


### 1.b. LSTM Model:

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_layer):
        super().__init__()
        self.embedding = embedding_layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

lstm_model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer).to(device)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

# Training Loop for LSTM:
for epoch in range(num_epochs):
    lstm_model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        lstm_optimizer.zero_grad()
        outputs = lstm_model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        lstm_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"LSTM Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

LSTM Epoch 1/5, Loss: 4.6989
LSTM Epoch 2/5, Loss: 2.1799
LSTM Epoch 3/5, Loss: 0.7830
LSTM Epoch 4/5, Loss: 0.3465
LSTM Epoch 5/5, Loss: 0.2169


In [ ]:
# Text Generation (RNN / LSTM):# Build id_to_token mapping (you need this for decoding)
id_to_token = {idx: tok for tok, idx in vocab.items()}

def encode_seed(seed_text, vocab, max_len=None):
    """
    Tokenize seed text the same way as training, then map to ids with <unk> fallback.
    We DO NOT auto-add <eos>/<sos> boundaries here; we just start from the seed.
    """
    # same tokenization pattern you used (excluding '*' as a token)
    seed_tokens = re.findall(r"\w+|[^\w\s*]", seed_text)

    # optional: start with <sos> if you trained with it
    if "<sos>" in vocab:
        seed_tokens = ["<sos>"] + seed_tokens

    unk_id = vocab["<unk>"]
    seed_ids = [vocab.get(t, unk_id) for t in seed_tokens]

    if max_len is not None:
        seed_ids = seed_ids[-max_len:]  # keep last max_len tokens
    return seed_ids, seed_tokens

@torch.no_grad()
def generate_text(
    model,
    vocab,
    id_to_token,
    seed_text="Alice",
    seq_length=50,
    max_new_tokens=120,
    temperature=1.0,
    top_k=30,
    device="cpu",
    stop_on_eos=True
):
    """
    Generates text by repeatedly predicting the next token.
    - temperature: higher => more random, lower => more greedy
    - top_k: sample only from the top_k most likely tokens (set to None to disable)
    """
    model.eval()

    # Encode seed
    seed_ids, _ = encode_seed(seed_text, vocab, max_len=seq_length)
    generated_ids = list(seed_ids)

    eos_id = vocab.get("<eos>", None)

    for _ in range(max_new_tokens):
        # Take last seq_length tokens as context
        context_ids = generated_ids[-seq_length:]
        x = torch.tensor(context_ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, seq_len)

        logits = model(x)                      # (1, seq_len, vocab_size)
        next_logits = logits[0, -1, :]         # (vocab_size,)

        # Temperature
        next_logits = next_logits / max(temperature, 1e-8)

        # Top-k filtering (optional)
        if top_k is not None and top_k > 0:
            top_vals, top_idx = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)))
            probs = F.softmax(top_vals, dim=-1)
            next_id = top_idx[torch.multinomial(probs, num_samples=1)].item()
        else:
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

        generated_ids.append(next_id)

        if stop_on_eos and eos_id is not None and next_id == eos_id:
            break

    # Decode ids -> tokens
    tokens = [id_to_token[i] for i in generated_ids]

    # Simple detokenization: add spaces between words, but not before punctuation
    out = []
    no_space_before = {".", ",", "!", "?", ":", ";", ")", "]", "}", "\"", "'"}
    no_space_after = {"(", "[", "{", "\"", "'"}
    prev = None
    for t in tokens:
        if t in {"<sos>", "<eos>", "<pad>"}:
            continue
        if not out:
            out.append(t)
        else:
            if t in no_space_before:
                out.append(t)
            elif prev in no_space_after:
                out.append(t)
            else:
                out.append(" " + t)
        prev = t

    return "".join(out)


# Example usage

print("---- RNN generation ----")
print(generate_text(
    model=model,
    vocab=vocab,
    id_to_token=id_to_token,
    seed_text="Alice was",
    seq_length=50,
    max_new_tokens=150,
    temperature=0.9,
    top_k=40,
    device=device,
    stop_on_eos=False   # set True if you want it to stop at <eos>
))

print("\n---- LSTM generation ----")
print(generate_text(
    model=lstm_model,
    vocab=vocab,
    id_to_token=id_to_token,
    seed_text="Alice was",
    seq_length=50,
    max_new_tokens=150,
    temperature=0.9,
    top_k=40,
    device=device,
    stop_on_eos=False
))



---- RNN generation ----
Alice was marked Advice looked flavour IV still nervous considering START talking golden decided you shelves fear OF Presently pop red knelt other mouse looked have plenty jar heads thump burning pocket dipped) she locked funny Either telescope lock air No _through_ still so larger _was_ Who OF miles) funny really Longitude opportunity pocket XI bottle put up leaves s curious _very_, an Little tiny did pop people before lock dream took rather: curious fear indeed once likely hear wondering begin knife couldn marked saw was go After or <unk> drop bit noticed much burning name away fifteen funny field fitted wouldn once but heap wise _never_ without whiskers way re! Now dark sort lit altogether went _would_ thing X ventured later forgotten Why beautifully earnestly littl words opened bright bleeds hot blown people thought looked locks from custard couldn more hot see feeling marked other <unk>

---- LSTM generation ----
Alice was nothing so _very_ remarkable in t

### Analysis:

RNN:
The basic RNN is able to learn short-term word relationships, but it struggles to keep track of information over longer parts of the text. As a result, the generated sentences often become fragmented and lose overall coherence.

LSTM:
The LSTM model produces longer and more coherent passages. It is better at maintaining the structure of the source text because its gating mechanism helps preserve important information across longer sequences.

* One thing to note is we are using a small subset of the book here (10k chars) and a sliding window dataset, so the model can partially memorize. That’s why losses can get very low and generation may copy phrases.

Now we want to compare models with different hyperparameters:

In [3]:
import random
import numpy as np

def train_model(model, dataloader, vocab_size, optimizer, criterion, device, num_epochs=5, clip_norm=1.0):
    model.to(device)
    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()

            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        epoch_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    return epoch_losses


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
def run_one_experiment(model_type, seq_length, batch_size, embedding_dim, hidden_dim, lr, num_epochs, seed_text="Alice was", seed= 0):
    set_seed(seed)
    # DataLoader depends on seq_length & batch_size
    dataset = TextDataset(encoded_text, seq_length=seq_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    vocab_size = len(vocab)
    criterion = nn.CrossEntropyLoss()

    # Fresh model each time (fair comparison)
    if model_type.lower() == "rnn":
        emb = nn.Embedding(vocab_size, embedding_dim)
        model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, emb)
    else:
        emb = nn.Embedding(vocab_size, embedding_dim)
        model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_dim, emb)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"\n--- {model_type.upper()} | seq={seq_length}, bs={batch_size}, emb={embedding_dim}, hid={hidden_dim}, lr={lr} ---")
    train_model(model, dataloader, vocab_size, optimizer, criterion, device, num_epochs=num_epochs)

    sample = generate_text(
        model=model,
        vocab=vocab,
        id_to_token=id_to_token,
        seed_text=seed_text,
        seq_length=seq_length,
        max_new_tokens=120,
        temperature=0.9,
        top_k=30,
        device=device,
        stop_on_eos=False
    )
    print("\nSample:\n", sample[:600], "...\n")
    return sample

In [6]:
# Investigate the impact of seq_len:
run_one_experiment("lstm", seq_length=30, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)
run_one_experiment("lstm", seq_length=80, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)

NameError: name 'LSTMLanguageModel' is not defined

In [5]:
# Investigate the impact of hidden_dim:
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=128, lr=1e-3, num_epochs=5)
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)

NameError: name 'LSTMLanguageModel' is not defined

In [78]:
# Investigate the impact of learning_rate:
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=3e-4, num_epochs=5)
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)


--- LSTM | seq=50, bs=32, emb=128, hid=256, lr=0.0003 ---
Epoch 1/5, Loss: 5.6653
Epoch 2/5, Loss: 4.5705
Epoch 3/5, Loss: 3.7464
Epoch 4/5, Loss: 2.9574
Epoch 5/5, Loss: 2.2404

Sample:
 Alice was not a hurry. ” (Alice no she to, “ in a moment, “ after on, ” (she said, this time she had upon a mouse, and behind as the fall passage not. There ’ to get think and about was coming door, but she jumped that a book to of through the house! ” thought Alice was! I should to be shutting herself it, she could for the hot as it was, and there that a little door she saw key to curtsey of the right did you you ever a moment to do: once she was ...


--- LSTM | seq=50, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.7857
Epoch 2/5, Loss: 2.4695
Epoch 3/5, Loss: 0.9747
Epoch 4/5, Loss: 0.4207
Epoch 5/5, Loss: 0.2506

Sample:
 Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the 

'Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, it might belong to one of the doors of the doors of the hall; but, alas! either the locks were too large, or the key was too small, but at it all seemed quite natural); but when the Rabbit actually _took a watch out of its waistcoat - pocket_, and looked at it, and then hurried on, Alice started to her feet'

I tested training hyperparameters by changing seq_length, hidden_dim, and learning rate while keeping the dataset and model structure the same. Increasing seq_length from 30 to 80 improved both the training loss and the quality of generated text, likely because the model had access to longer context. Increasing hidden_dim from 128 to 256 significantly improved fluency and reduced repetition, which shows that model capacity matters for text generation. For learning rate, lr=0.0003 underfit within 5 epochs and produced weak text, while lr=0.001 converged faster and produced more coherent outputs.

In [82]:
def plot_two_loss_curves(losses1, losses2, label1="RNN", label2="LSTM", title="Loss Curves Comparison"):
    plt.figure()
    plt.plot(range(1, len(losses1) + 1), losses1, marker="o", label=label1)
    plt.plot(range(1, len(losses2) + 1), losses2, marker="o", label=label2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [83]:
def plot_loss_curve(losses, title="Training Loss Curve"):
    plt.figure()
    plt.plot(range(1, len(losses) + 1), losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True)
    plt.show()

In [84]:
rnn_losses, rnn_sample = run_one_experiment(
    model_type="rnn", seq_length=50, batch_size=32,
    embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5,
    seed_text="Alice was", seed=0
)

lstm_losses, lstm_sample = run_one_experiment(
    model_type="lstm", seq_length=50, batch_size=32,
    embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5,
    seed_text="Alice was", seed=0
)

plot_two_loss_curves(rnn_losses, lstm_losses, label1="RNN", label2="LSTM",
                     title="RNN vs LSTM Training Loss (seq_length=50)")


--- RNN | seq=50, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 3.9831
Epoch 2/5, Loss: 1.3110
Epoch 3/5, Loss: 0.3946
Epoch 4/5, Loss: 0.1918
Epoch 5/5, Loss: 0.1280

Sample:
 Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another it passage, and the fall _never_ come to an end? “ I wonder how many miles I ’ ve fallen by this time? ” she said aloud. “ I must be getting somewhere near the centre of the earth. Let me see: that would be four thousand miles down, for a jar! ” thought Alice to herself, “ after such a fall as this, I shall think nothing ...



ValueError: too many values to unpack (expected 2)

In [85]:
loss_30, _ = run_one_experiment("lstm", 30, 32, 128, 256, 1e-3, 5, seed_text="Alice was", seed=0)
loss_50, _ = run_one_experiment("lstm", 50, 32, 128, 256, 1e-3, 5, seed_text="Alice was", seed=0)
loss_80, _ = run_one_experiment("lstm", 80, 32, 128, 256, 1e-3, 5, seed_text="Alice was", seed=0)

plt.figure()
plt.plot(range(1, 6), loss_30, marker="o", label="seq_length=30")
plt.plot(range(1, 6), loss_50, marker="o", label="seq_length=50")
plt.plot(range(1, 6), loss_80, marker="o", label="seq_length=80")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Loss Curves for Different seq_length Values")
plt.legend()
plt.grid(True)
plt.show()


--- LSTM | seq=30, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.8873
Epoch 2/5, Loss: 2.7668
Epoch 3/5, Loss: 1.2679
Epoch 4/5, Loss: 0.6056
Epoch 5/5, Loss: 0.3706

Sample:
 Alice was not a bit hurt, and she jumped up key to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it might end, as she couldn ’ t answer either, it didn ’ t sound at all the right word) “ — but I shall have to ask them what the name of the country, you know. Please, Ma ’ am, is this New Zealand or Australia? ” (and she tried to curtsey as she spoke — fancy _curtseying_ as you ’ ...



ValueError: too many values to unpack (expected 2)

the plotting tbd

# 2. Seq2Seq Machine Translation with Attention

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import random
import re

## 2.1 Dataset Preparation and Preprocessing

In [ ]:
!wget -q http://www.manythings.org/anki/fra-eng.zip
!unzip -q fra-eng.zip
!ls -lh

--2026-03-07 17:19:40--  http://www.manythings.org/anki/fra-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8242175 (7.9M) [application/zip]
Saving to: ‘fra-eng.zip.3’

fra-eng.zip.3       100%[===================>]   7.86M  15.0MB/s    in 0.5s    

2026-03-07 17:19:41 (15.0 MB/s) - ‘fra-eng.zip.3’ saved [8242175/8242175]

Archive:  fra-eng.zip
replace _about.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: total 66M
-rw-r--r-- 1 root root 1.5K Feb 13 06:17 _about.txt
-rw-r--r-- 1 root root 7.9M Feb 13 00:33 fra-eng.zip
-rw-r--r-- 1 root root 7.9M Feb 13 00:33 fra-eng.zip.1
-rw-r--r-- 1 root root 7.9M Feb 13 00:33 fra-eng.zip.2
-rw-r--r-- 1 root root 7.9M Feb 13 00:33 fra-eng.zip.3
-rw-r--r-- 1 root root  35M Feb 13 06:17 fra.txt
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


In [11]:
# Load Sentences:

with open("/content/fra.txt", encoding="utf-8") as f:
    lines = f.read().splitlines()

print(len(lines))
print(lines[0])

pairs = [] # list of (english, french) tuples
for line in lines:
    parts = line.split("\t")
    if len(parts) >= 2:
        eng, fra = parts[0].strip(), parts[1].strip()
        pairs.append((eng, fra))

print("Original Number of Pairs:", len(pairs))
print("5 first rows in pairs: ", pairs[:5])

#limiting the length on input pairs (shuffling, because the first pairs are simpler sentences, 
# and we want a mixture of simple and complex sentences)
random.shuffle(pairs)
pairs = pairs[:5000]


# Tokenization:
def tokenize_sentence(sentence):
    sentence = sentence.lower().strip()
    return re.findall(r"[a-zA-ZÀ-ÿ']+|[.,!?;:]", sentence)

tokenized_pairs = [] # list of (english tokens, french tokens) tuples
for eng, fra in pairs:
    eng_tokens = tokenize_sentence(eng)
    fra_tokens = tokenize_sentence(fra)
    tokenized_pairs.append((eng_tokens, fra_tokens))
print("first 5 tokenized pairs:", tokenized_pairs[:10])

tokenized_english = [token[0] for token in tokenized_pairs]
tokenized_french = [token[1] for token in tokenized_pairs]


# Build Vocabularies:
eng_vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
fra_vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}

def build_vocab(tokenized_sentences, vocab):
    idx = len(vocab)
    for tokens in tokenized_sentences:
            for tok in tokens:
                if tok not in vocab:
                    vocab[tok] = idx
                    idx += 1
    return vocab
eng_vocab = build_vocab(tokenized_english, eng_vocab)
fra_vocab = build_vocab(tokenized_french,  fra_vocab)

print("English vocab size:", len(eng_vocab))
print("French vocab size:", len(fra_vocab))


def sentence_to_tensor(tokens, vocab):
    ids = [vocab["<sos>"]]

    for token in tokens:
        ids.append(vocab.get(token, vocab["<unk>"]))

    ids.append(vocab["<eos>"])

    return torch.tensor(ids, dtype=torch.long)

eng_tensors = [sentence_to_tensor(tokens, eng_vocab) for tokens in tokenized_english]
fra_tensors = [sentence_to_tensor(tokens, fra_vocab) for tokens in tokenized_french]


def pad_sentences(tensor_list, vocab):
    pad_id = vocab["<pad>"]
    max_len = max(len(t) for t in tensor_list)

    padded_tensors = []
    for t in tensor_list:
        padded = t.tolist()  # tensor -> python list of ints
        while len(padded) < max_len:
            padded.append(pad_id)
        padded_tensors.append(torch.tensor(padded, dtype=torch.long))

    return padded_tensors, max_len

eng_tensors, eng_max_len = pad_sentences(eng_tensors, eng_vocab)
fra_tensors, fra_max_len = pad_sentences(fra_tensors, fra_vocab)

print("Max English length after <sos>/<eos>:", eng_max_len)
print("Max French length after <sos>/<eos>:", fra_max_len)
print("Example padded English tensor shape:", eng_tensors[0].shape)
print("Example padded French tensor shape:", fra_tensors[0].shape)


class TranslationDataset(Dataset):
    def __init__(self, src_tensors, tgt_tensors):
        self.src = src_tensors
        self.tgt = tgt_tensors

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

dataset = TranslationDataset(eng_tensors, fra_tensors)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Test one batch:
src_batch, tgt_batch = next(iter(dataloader))
print("Source batch shape:", src_batch.shape)  # (batch_size, max_src_len)
print("Target batch shape:", tgt_batch.shape)  # (batch_size, max_tgt_len)
print("Source batch dtype:", src_batch.dtype)  # should be torch.int64

240521
Go.	Va !	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1158250 (Wittydev)
Original Number of Pairs: 240521
5 first rows in pairs:  [('Go.', 'Va !'), ('Go.', 'Marche.'), ('Go.', 'En route !'), ('Go.', 'Bouge !'), ('Hi.', 'Salut !')]
first 5 tokenized pairs: [(['she', 'is', 'fond', 'of', 'singing', 'old', 'songs', '.'], ['elle', 'adore', 'chanter', 'de', 'vieilles', 'chansons', '.']), (['that', 'vending', 'machine', 'is', 'out', 'of', 'order', '.'], ['ce', 'distributeur', 'automatique', 'est', 'hors', 'service', '.']), (['your', 'sweater', 'is', 'inside', 'out', '.'], ['ton', 'pull', 'est', 'à', "l'envers", '.']), (['what', 'did', 'you', 'make', '?'], ['que', 'faisais', 'tu', '?']), (['do', 'you', 'recognize', 'me', '?'], ['me', 'reconnais', 'tu', '?']), (['i', 'know', 'some', 'french', '.'], ['je', 'connais', 'un', 'peu', 'le', 'français', '.']), (["i'm", 'not', 'satisfied', 'with', 'your', 'performance', '.'], ['je', 'ne', 'suis', 'pas', 'satisfaite', 'de', 'votre

## 2.2 Seq-to-seq model with Attention

In [12]:
# Encoder:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)  # combine forward + backward states

    def forward(self, src):
        # src: (batch_size, src_len)
        embedded = self.embedding(src)  # (batch_size, src_len, emb_dim)

        # outputs: (batch_size, src_len, hid_dim * 2)
        # hidden: (2, batch_size, hid_dim)
        # cell:   (2, batch_size, hid_dim)
        outputs, (hidden, cell) = self.rnn(embedded)

        # combine forward and backward hidden states
        hidden = torch.tanh(self.fc(torch.cat((hidden[0], hidden[1]), dim=1)))  # (batch_size, hid_dim)
        hidden = hidden.unsqueeze(0)  # (1, batch_size, hid_dim)

        # combine forward and backward cell states
        cell = torch.tanh(self.fc(torch.cat((cell[0], cell[1]), dim=1)))  # (batch_size, hid_dim)
        cell = cell.unsqueeze(0)  # (1, batch_size, hid_dim)

        return outputs, hidden, cell


# Attention
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v = nn.Parameter(torch.rand(hidden_dim))

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch_size, 1, hidden_dim)
        # encoder_outputs: (batch_size, src_len, hidden_dim * 2)

        seq_len = encoder_outputs.shape[1]

        # repeat decoder hidden state src_len times
        hidden = hidden.repeat(1, seq_len, 1)  # (batch_size, src_len, hidden_dim)

        # concatenate hidden state and encoder outputs
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        # energy: (batch_size, src_len, hidden_dim)

        # compute attention scores
        attention = torch.sum(self.v * energy, dim=2)  # (batch_size, src_len)

        return F.softmax(attention, dim=1)


# Decoder with Attention
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = Attention(hidden_dim)
        self.rnn = nn.LSTM(hidden_dim * 2 + emb_dim, hidden_dim, batch_first=True)

        # final output layer
        self.fc = nn.Linear(hidden_dim + hidden_dim * 2 + emb_dim, output_dim)

    def forward(self, x, hidden, cell, encoder_outputs):
        # x: (batch_size,)
        # hidden: (1, batch_size, hidden_dim)
        # cell: (1, batch_size, hidden_dim)
        # encoder_outputs: (batch_size, src_len, hidden_dim * 2)

        x = x.unsqueeze(1)  # (batch_size, 1)
        embedded = self.embedding(x)  # (batch_size, 1, emb_dim)

        # attention weights
        attention_weights = self.attention(hidden[-1].unsqueeze(1), encoder_outputs)
        # attention_weights: (batch_size, src_len)

        # weighted context vector
        weighted = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs)
        # weighted: (batch_size, 1, hidden_dim * 2)

        # input to decoder LSTM
        rnn_input = torch.cat((embedded, weighted), dim=2)
        # rnn_input: (batch_size, 1, emb_dim + hidden_dim * 2)

        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        # output: (batch_size, 1, hidden_dim)

        # final prediction
        prediction = self.fc(
            torch.cat((output.squeeze(1), weighted.squeeze(1), embedded.squeeze(1)), dim=1)
        )
        # prediction: (batch_size, output_dim)

        return prediction, hidden, cell

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters:
input_dim = len(eng_vocab)     # English vocab size
output_dim = len(fra_vocab)    # French vocab size
emb_dim = 64
hidden_dim = 128
epochs = 10
learning_rate = 0.001
teacher_forcing_ratio = 0.5
clip = 1.0
pad_idx = fra_vocab["<pad>"]

# Model Initialization
encoder = Encoder(input_dim, emb_dim, hidden_dim).to(device)
decoder = Decoder(output_dim, emb_dim, hidden_dim).to(device)

# Loss and Optimizers
criterion = nn.CrossEntropyLoss(ignore_index=fra_vocab["<pad>"])
encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(epochs):
    encoder.train()
    decoder.train()

    total_loss = 0

    for src, tgt in dataloader:
        src = src.to(device)   # (batch_size, src_len)
        tgt = tgt.to(device)   # (batch_size, tgt_len)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # Encoder forward
        encoder_outputs, hidden, cell = encoder(src)

        # First decoder input = <sos>
        decoder_input = tgt[:, 0]   # shape: (batch_size,)

        loss = torch.tensor(0.0, device=device)
        tgt_len = tgt.shape[1]
        valid_steps = 0

        for t in range(1, tgt_len):
            output, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)
            # output: (batch_size, output_dim)
            if (tgt[:, t] != pad_idx).any():
                loss += criterion(output, tgt[:, t])
                valid_steps += 1

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            decoder_input = tgt[:, t] if teacher_force else top1
        
        if valid_steps > 0:
            loss = loss / valid_steps
        loss.backward()

        torch.nn.utils.clip_grad_norm_(encoder.parameters(), clip)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), clip)

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(dataloader):.4f}")

Epoch 1/10, Loss: 5.0134
Epoch 2/10, Loss: 4.3713
Epoch 3/10, Loss: 4.0786
Epoch 4/10, Loss: 3.7787
Epoch 5/10, Loss: 3.4525
Epoch 6/10, Loss: 3.0999
Epoch 7/10, Loss: 2.7611
Epoch 8/10, Loss: 2.3920
Epoch 9/10, Loss: 2.1227
Epoch 10/10, Loss: 1.8944


In [14]:
# Reverse French vocab: id -> token
fra_id_to_token = {idx: tok for tok, idx in fra_vocab.items()}

def sentence_to_tensor_inference(sentence, vocab):
    tokens = tokenize_sentence(sentence)
    ids = [vocab["<sos>"]]

    for token in tokens:
        ids.append(vocab.get(token, vocab["<unk>"]))

    ids.append(vocab["<eos>"])

    return torch.tensor(ids, dtype=torch.long)

@torch.no_grad()
def translate_sentence(sentence, max_len=20):
    encoder.eval()
    decoder.eval()

    src_tensor = sentence_to_tensor_inference(sentence, eng_vocab).unsqueeze(0).to(device)
    # shape: (1, src_len)

    encoder_outputs, hidden, cell = encoder(src_tensor)

    decoder_input = torch.tensor([fra_vocab["<sos>"]], device=device)
    translated_tokens = []

    for _ in range(max_len):
        output, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)
        pred_token = output.argmax(1).item()

        if pred_token == fra_vocab["<eos>"]:
            break

        translated_tokens.append(fra_id_to_token[pred_token])
        decoder_input = torch.tensor([pred_token], device=device)

    return " ".join(translated_tokens)

In [15]:
print(translate_sentence("i am a student"))
print(translate_sentence("he is my friend"))
print(translate_sentence("we love music"))

je suis désolée un peu un bon .
il est l'ami mon domaine est poussiéreux .
nous avons nous nous nous nous nous nous nous nous .


## 2.3 Seq-to-seq model without Attention

In [ ]:
# Decoder WITHOUT Attention
class Decoder_NoAttention(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(output_dim, emb_dim)

        # input is only embedding (no attention context)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden, cell, encoder_outputs=None):

        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell
    
# Train model WITHOUT attention
decoder_noatt = Decoder_NoAttention(output_dim, emb_dim, hidden_dim).to(device)

encoder_noatt = Encoder(input_dim, emb_dim, hidden_dim).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=fra_vocab["<pad>"])

enc_opt = optim.Adam(encoder_noatt.parameters(), lr=learning_rate)
dec_opt = optim.Adam(decoder_noatt.parameters(), lr=learning_rate)

epochs = 5   # keeping small so it runs fast

# Training Loop
for epoch in range(epochs):

    total_loss = 0
    counted_batches = 0

    for src, tgt in dataloader:

        src = src.to(device)
        tgt = tgt.to(device)

        enc_opt.zero_grad()
        dec_opt.zero_grad()

        encoder_outputs, hidden, cell = encoder_noatt(src)

        decoder_input = tgt[:,0] # <sos>

        loss = torch.tensor(0.0, device=device)
        tgt_len = tgt.shape[1]
        valid_steps = 0

        for t in range(1, tgt_len):
            # Skip loss calculation if all tokens in this position are <pad>
            if (tgt[:, t] != pad_idx).sum().item() == 0:
                continue

            output, hidden, cell = decoder_noatt(decoder_input, hidden, cell)

            loss += criterion(output, tgt[:,t])
            valid_steps += 1

            teacher_force = random.random() < teacher_forcing_ratio

            top1 = output.argmax(1)

            decoder_input = tgt[:,t] if teacher_force else top1

        # Only backprop if batch has at least one valid target step
        if valid_steps > 0:
            loss = loss / valid_steps
            loss.backward()

            torch.nn.utils.clip_grad_norm_(encoder_noatt.parameters(), clip)
            torch.nn.utils.clip_grad_norm_(decoder_noatt.parameters(), clip)

            enc_opt.step()
            dec_opt.step()

            total_loss += loss.item()
            counted_batches += 1

    epoch_loss = total_loss / counted_batches if counted_batches > 0 else 0.0
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")


# Translation function for no-attention model

# Reverse French vocab: id -> token
fra_id_to_token = {idx: tok for tok, idx in fra_vocab.items()}

@torch.no_grad()
def translate_no_attention(sentence, max_len=20):
    encoder_noatt.eval()
    decoder_noatt.eval()

    src_tensor = sentence_to_tensor_inference(sentence, eng_vocab).unsqueeze(0).to(device)

    encoder_outputs, hidden, cell = encoder_noatt(src_tensor)

    decoder_input = torch.tensor([fra_vocab["<sos>"]], dtype=torch.long, device=device)

    translated_tokens = []

    for _ in range(max_len):
        output, hidden, cell = decoder_noatt(decoder_input, hidden, cell)

        pred_token = output.argmax(1).item()

        if pred_token == fra_vocab["<eos>"]:
            break

        translated_tokens.append(fra_id_to_token.get(pred_token, "<unk>"))
        decoder_input = torch.tensor([pred_token], dtype=torch.long, device=device)

    return " ".join(translated_tokens)

Epoch 1/5, Loss: 7.3006
Epoch 2/5, Loss: 6.1869
Epoch 3/5, Loss: 6.0696
Epoch 4/5, Loss: 5.9149
Epoch 5/5, Loss: 5.8087


Loss values decrease obviously compared to the model with attention.

In [23]:
# Qualitative comparison between attention and no-attention models:
test_sentences = [
    "i am a student",
    "he is my friend",
    "we love music"
]

for s in test_sentences:

    print("EN:", s)

    print("With Attention:", translate_sentence(s))

    print("Without Attention:", translate_no_attention(s))

    print()

EN: i am a student
With Attention: je suis désolée un peu un bon .
Without Attention: je ne que tu ne pas pas , je ne pas pas ?

EN: he is my friend
With Attention: il est l'ami mon domaine est poussiéreux .
Without Attention: je ne que tu que je ne pas pas , je ne pas pas . .

EN: we love music
With Attention: nous avons nous nous nous nous nous nous nous nous .
Without Attention: je ne que tu que je ne pas pas ?



## 2.4. BLEU Evaluation Section

In [ ]:
!pip install nltk

In [26]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

In [27]:
test_samples = pairs[100:120]   # 20 examples for evaluation

smooth = SmoothingFunction().method1

# Select some test sentence pairs
test_samples = pairs[100:120]   # you can change this range if you want

references = []
predictions_with_attention = []
predictions_without_attention = []

print("Sample Translation Comparison:\n")

for eng, fra in test_samples:
    # Reference translation tokens
    ref_tokens = tokenize_sentence(fra)
    references.append([ref_tokens])   # BLEU expects list of reference lists

    # Prediction with attention
    pred_att = translate_sentence(eng)
    pred_att_tokens = tokenize_sentence(pred_att)
    predictions_with_attention.append(pred_att_tokens)

    # Prediction without attention
    pred_noatt = translate_no_attention(eng)
    pred_noatt_tokens = tokenize_sentence(pred_noatt)
    predictions_without_attention.append(pred_noatt_tokens)

    print("EN:", eng)
    print("REF:", fra)
    print("PRED (With Attention):", pred_att)
    print("PRED (Without Attention):", pred_noatt)
    print("-" * 80)

# Corpus BLEU scores
bleu_with_attention = corpus_bleu(
    references,
    predictions_with_attention,
    smoothing_function=smooth
)

bleu_without_attention = corpus_bleu(
    references,
    predictions_without_attention,
    smoothing_function=smooth
)

print("\nFinal BLEU Comparison:")
print(f"Corpus BLEU Score (With Attention): {bleu_with_attention:.4f}")
print(f"Corpus BLEU Score (Without Attention): {bleu_without_attention:.4f}")

Sample Translation Comparison:

EN: I was thrown off guard.
REF: J'ai été pris par surprise.
PRED (With Attention): j'ai été un peu de j'ai bu .
PRED (Without Attention): je ne que tu que je ne pas pas , je ne pas pas . .
--------------------------------------------------------------------------------
EN: I need your number.
REF: J'ai besoin de ton numéro.
PRED (With Attention): je besoin besoin de votre numéro de votre numéro .
PRED (Without Attention): je ne que tu que je ne pas pas , je ne pas pas . .
--------------------------------------------------------------------------------
EN: I'd like to rent your most inexpensive car for a week.
REF: J'aimerais louer votre voiture la moins chère, pour une semaine.
PRED (With Attention): j'aimerais louer de de que pour une semaine de semaine de semaine de semaine .
PRED (Without Attention): je ne que tu que je ne pas pas , je ne pas pas . .
--------------------------------------------------------------------------------
EN: I almost never u

## 2.5. Experimenting hyperparameters

In [ ]:
# Hyperparameter Experiments (on the attention-based model):

def train_model_for_experiment(hidden_dim=128, learning_rate=0.001, batch_size=32, epochs=3):
    # New dataloader for this experiment
    dataloader_exp = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # New model for this experiment
    encoder_exp = Encoder(len(eng_vocab), emb_dim=64, hid_dim=hidden_dim).to(device)
    decoder_exp = Decoder(len(fra_vocab), emb_dim=64, hidden_dim=hidden_dim).to(device)

    # Loss and optimizers
    pad_idx = fra_vocab["<pad>"]
    criterion_exp = nn.CrossEntropyLoss(ignore_index=pad_idx)
    encoder_optimizer_exp = optim.Adam(encoder_exp.parameters(), lr=learning_rate)
    decoder_optimizer_exp = optim.Adam(decoder_exp.parameters(), lr=learning_rate)

    epoch_losses = []

    for epoch in range(epochs):
        encoder_exp.train()
        decoder_exp.train()

        total_loss = 0
        num_batches = 0

        for src, tgt in dataloader_exp:
            src = src.to(device)
            tgt = tgt.to(device)

            encoder_optimizer_exp.zero_grad()
            decoder_optimizer_exp.zero_grad()

            encoder_outputs, hidden, cell = encoder_exp(src)
            decoder_input = tgt[:, 0]

            loss = 0
            valid_steps = 0
            tgt_len = tgt.shape[1]

            for t in range(1, tgt_len):
                output, hidden, cell = decoder_exp(decoder_input, hidden, cell, encoder_outputs)

                if (tgt[:, t] != pad_idx).any():
                    loss += criterion_exp(output, tgt[:, t])
                    valid_steps += 1

                teacher_force = random.random() < 0.5
                top1 = output.argmax(1)
                decoder_input = tgt[:, t] if teacher_force else top1

            if valid_steps > 0:
                loss = loss / valid_steps
                loss.backward()

                torch.nn.utils.clip_grad_norm_(encoder_exp.parameters(), 1.0)
                torch.nn.utils.clip_grad_norm_(decoder_exp.parameters(), 1.0)

                encoder_optimizer_exp.step()
                decoder_optimizer_exp.step()

                total_loss += loss.item()
                num_batches += 1

        avg_loss = total_loss / num_batches if num_batches > 0 else 0
        epoch_losses.append(avg_loss)

    return epoch_losses


# Different hyperparameter settings to compare
experiments = [
    {"hidden_dim": 64,  "learning_rate": 0.001,  "batch_size": 32, "epochs": 3},
    {"hidden_dim": 128, "learning_rate": 0.001,  "batch_size": 32, "epochs": 3},
    {"hidden_dim": 128, "learning_rate": 0.0005, "batch_size": 32, "epochs": 3},
    {"hidden_dim": 128, "learning_rate": 0.001,  "batch_size": 64, "epochs": 3},
]

results = []

for i, exp in enumerate(experiments, 1):
    print(f"Running Experiment {i}: {exp}")

    losses = train_model_for_experiment(
        hidden_dim=exp["hidden_dim"],
        learning_rate=exp["learning_rate"],
        batch_size=exp["batch_size"],
        epochs=exp["epochs"]
    )

    result = {
        "Experiment": i,
        "Hidden Dim": exp["hidden_dim"],
        "Learning Rate": exp["learning_rate"],
        "Batch Size": exp["batch_size"],
        "Epochs": exp["epochs"],
        "Final Loss": losses[-1]
    }

    results.append(result)
    print(f"Final Loss: {losses[-1]:.4f}\n")


# Print results clearly
print("Hyperparameter Comparison Results:")
for r in results:
    print(r)

Running Experiment 1: {'hidden_dim': 64, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 3}
Final Loss: 4.2320

Running Experiment 2: {'hidden_dim': 128, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 3}
Final Loss: 4.0845

Running Experiment 3: {'hidden_dim': 128, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 3}
Final Loss: 4.3678

Running Experiment 4: {'hidden_dim': 128, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 3}
Final Loss: 4.3132

Hyperparameter Comparison Results:
{'Experiment': 1, 'Hidden Dim': 64, 'Learning Rate': 0.001, 'Batch Size': 32, 'Epochs': 3, 'Final Loss': 4.231970589631682}
{'Experiment': 2, 'Hidden Dim': 128, 'Learning Rate': 0.001, 'Batch Size': 32, 'Epochs': 3, 'Final Loss': 4.084486124621835}
{'Experiment': 3, 'Hidden Dim': 128, 'Learning Rate': 0.0005, 'Batch Size': 32, 'Epochs': 3, 'Final Loss': 4.367834079037806}
{'Experiment': 4, 'Hidden Dim': 128, 'Learning Rate': 0.001, 'Batch Size': 64, 'Epochs': 3, 'Final Loss': 4.31316409231741

## Report of "Seq2Seq Machine Translation with Attention" section:

In this part, I implemented a Seq2Seq machine translation model with attention for English-to-French translation. I first prepared the dataset by loading sentence pairs, tokenizing both languages, converting the tokens into integer sequences, and padding them to the same length. Since the original dataset was very large, I randomly shuffled the sentence pairs and used 5000 of them to keep the training time reasonable while still having a mix of simple and more complex sentences.

### Comparison between the model with Attention and without Attention:
For the model, I used an LSTM-based encoder and decoder and added an attention mechanism between them. The attention model was trained for 10 epochs, and the training loss decreased from 5.0134 to 1.8944, which shows that the model was learning during training. I also trained a second version without attention for comparison. That model had much higher loss values, decreasing only from 7.3006 to 5.8087 after 5 epochs, which already suggests weaker performance.

### Translation results:
When I tested both models on sample sentences, the attention-based model produced outputs that were still imperfect but often closer to the correct meaning. It was able to capture some important words and phrases from the input, even though many outputs still had repetition or grammatical mistakes. For example, in some cases it correctly generated key words such as numéro, ordinateur, or Mont Fuji. In contrast, the model without attention mostly generated repetitive and unrelated sentence patterns, showing that it was not able to use the source sentence effectively.


### BLEU score comparison:
The BLEU score also confirmed this difference. The attention-based model achieved a corpus BLEU score of 0.1274, while the model without attention only achieved 0.0014. Even though 0.1274 is still a low BLEU score, it is reasonable for a small assignment model trained on only 5000 sentence pairs and a limited number of epochs. The important point is that the model with attention performed much better than the one without attention, both qualitatively and quantitatively. This shows that the attention mechanism helped the decoder focus on more relevant parts of the input sentence instead of relying only on a single compressed representation from the encoder.

### Hyperparameter test:
I also tested a few hyperparameter settings on the attention-based model:
Experiment 1: 
Hidden Dim: 64, Learning Rate: 0.001, Batch Size: 32, 
Final Loss: 4.231970589631682

Experiment 2: 
Hidden Dim: 128, Learning Rate: 0.001, Batch Size: 32, 
Final Loss: 4.084486124621835

Experiment 3:
Hidden Dim: 128, Learning Rate: 0.0005, Batch Size: 32, 
Final Loss: 4.367834079037806

Experiment: 4:
Hidden Dim: 128, Learning Rate: 0.001, Batch Size: 64,
'Final Loss': 4.313164092317412

Among the tested cases, the best result was obtained with hidden dimension 128, learning rate 0.001, and batch size 32, which gave the lowest final loss of 4.0845 after 3 epochs.
Also the effects of different hyperparameter changes (by comparing a pair with only one hyperparameter changed, and others fixed) are as follows:
- Increasing the hidden size slightly improved the result
(This is likely because a larger hidden dimension allows the model to store richer representations of the input sentence. With more hidden units, the encoder can capture more semantic and syntactic information from the source sequence, which helps the decoder generate better predictions.)

- Lowering the learning rate (that makes the training slower), increased loss
(A smaller learning rate means that the optimizer updates the model parameters more slowly. Since the experiments were run only for a small number of epochs, the model with the lower learning rate did not have enough time to converge, resulting in a higher final loss.)

- Increasing the batch size gave a slightly higher loss.
(Larger batch sizes reduce the number of parameter updates per epoch and may make the optimization process less responsive to small variations in the data. In this experiment, the smaller batch size likely helped the model adjust its parameters more frequently, leading to slightly better convergence.)


Based on these experiments, the original setting of hidden size 128, learning rate 0.001, and batch size 32 was the most effective among the tested options.

Overall, the results show that the Seq2Seq model with attention worked better than the model without attention, but the translations were still far from perfect for this scale of model. The main problems of translation results were repetition, incomplete phrases, and grammatical errors. These issues are expected because the model was trained on a limited subset of the dataset and for a relatively small number of epochs. Still, the assignment clearly showed that attention improves translation quality and plays an important role in Seq2Seq machine translation.